In [1]:
%load_ext autoreload
%autoreload 2
### Set CUDA device
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [2]:
from omegaconf import OmegaConf

from tqdm import tqdm
sys.path.append('../')
from data_utils import get_data_module
from model.unet import get_unet_module

In [4]:
### Important Globals
DATASET = 'mnmv2' # or 'pmri'
UNET_CKPTS = {
    "mnmv2": 'mnmv2_symphony_dropout-0-1_2025-01-14-15-19', 
    'pmri': 'pmri_runmc_dropout-0-1_2025-01-14-15-58',
}

### configs
unet_cfg = OmegaConf.load('../configs/unet/monai_unet.yaml')
if DATASET == 'mnmv2':
    unet_cfg.out_channels = 4
    num_classes = 4
    data_cfg = OmegaConf.load('../configs/data/mnmv2.yaml')
    domain = 'Symphony'
else:
    unet_cfg.out_channels = 1
    num_classes = 2
    data_cfg = OmegaConf.load('../configs/data/pmri.yaml')
    domain = 'RUNMC'

data_cfg.dataset = DATASET
data_cfg.domain = domain
data_cfg.non_empty_target = True # ignore slice with empty ground truth mask

datamodule = get_data_module(
    cfg=data_cfg
)

# you can assign a checkpoint in the config file, or you can assign it here
ckpt = UNET_CKPTS[data_cfg.dataset]
unet_cfg.checkpoint_path = f'../../{unet_cfg.checkpoint_dir}{ckpt}.ckpt'
unet_cfg.dropout = 0.1


### load a unet
unet = get_unet_module(
    cfg=unet_cfg,
    metadata=OmegaConf.to_container(unet_cfg),
    load_from_checkpoint=True # set to True if you want to load the checkpoint
) # unet is a lightning module. If you need a python model, you can access it via unet.model


In [5]:
# load data. Before that, you only have a class object
setup = 'test' # or 'fit' or whatever is defined in the LightningDataModule setup function
datamodule.setup(setup)

In [ ]:
### access the data
data = datamodule.test_dataloader() # datamodule.train_dataloader()]

### now you have a dict with the dataloaders defined in the LightningDataModule